## 1. Cargar los datos

In [1]:
#Pathlib sirve para que todos podamos trabajar desde cualquier sistema operativo.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path().resolve().parent
DATA_PATH = ROOT / "data_raw"

print("Directorio actual:", ROOT)
print("Ruta de datos:", DATA_PATH)

Directorio actual: C:\Users\naila\OneDrive\Documentos\Nueva carpeta (2)\Projecto-2-Vanguard-ab-test
Ruta de datos: C:\Users\naila\OneDrive\Documentos\Nueva carpeta (2)\Projecto-2-Vanguard-ab-test\data_raw


In [2]:
#Descarga de datos otra vez.
df_demo        = pd.read_csv(DATA_PATH / "df_final_demo.txt")
df_experiment  = pd.read_csv(DATA_PATH / "df_final_experiment_clients.txt")
df_web_pt1     = pd.read_csv(DATA_PATH / "df_final_web_data_pt_1.txt")
df_web_pt2     = pd.read_csv(DATA_PATH / "df_final_web_data_pt_2.txt")

print("df_demo filas:", len(df_demo))
print("df_experiment filas:", len(df_experiment))
print("df_web_pt1 filas:", len(df_web_pt1))
print("df_web_pt2 filas:", len(df_web_pt2))

df_demo filas: 70609
df_experiment filas: 70609
df_web_pt1 filas: 343141
df_web_pt2 filas: 412264


## 2. Limpieza df_demo

In [3]:
# 2.1 Duplicados
print("Duplicados en df_demo:", df_demo.duplicated().sum())

Duplicados en df_demo: 0


In [4]:
# 2.2 Nulos
print("Nulos en df_demo:")
print(df_demo.isna().sum())

Nulos en df_demo:
client_id            0
clnt_tenure_yr      14
clnt_tenure_mnth    14
clnt_age            15
gendr               14
num_accts           14
bal                 14
calls_6_mnth        14
logons_6_mnth       14
dtype: int64


In [5]:
# 2.3 Eliminar nulos
df_demo = df_demo.dropna()
print("Filas antes de limpiar: 70609")
print("Filas después de limpiar:", len(df_demo))

Filas antes de limpiar: 70609
Filas después de limpiar: 70594


In [6]:
# 2.4 Revisión columna género. En el enunciado del ejercicio dice que necesitaremos ese dato.
print("Valores únicos en género:")
print(df_demo["gendr"].value_counts())

Valores únicos en género:
gendr
U    24122
M    23724
F    22745
X        3
Name: count, dtype: int64


In [7]:
# 2.5 Eliminar valores incorrectos en género (X)
df_demo = df_demo[df_demo["gendr"] != "X"]
print("Filas antes de limpiar género: 70594")
print("Filas después de limpiar género:", len(df_demo))
print()
print("Valores género tras limpieza:")
print(df_demo["gendr"].value_counts())

Filas antes de limpiar género: 70594
Filas después de limpiar género: 70591

Valores género tras limpieza:
gendr
U    24122
M    23724
F    22745
Name: count, dtype: int64


## 3. Limpieza df_experiment

In [8]:
# 3.1 Duplicados
print("Duplicados en df_experiment:", df_experiment.duplicated().sum())

Duplicados en df_experiment: 0


In [10]:
# 3.2 Nulos
print("Nulos en df_experiment:")
print(df_experiment.isna().sum())

Nulos en df_experiment:
client_id        0
Variation    20109
dtype: int64


In [11]:
# 3.3 Eliminar clientes sin grupo (Variation nulo)
df_experiment = df_experiment.dropna(subset=["Variation"])

print("Filas antes de limpiar: 70609")
print("Filas después de limpiar:", len(df_experiment))
print()
print("Distribución Test vs Control:")
print(df_experiment["Variation"].value_counts())

Filas antes de limpiar: 70609
Filas después de limpiar: 50500

Distribución Test vs Control:
Variation
Test       26968
Control    23532
Name: count, dtype: int64


## 4. Limpieza df_web

In [12]:
# 4.1 Duplicados
print("Duplicados en df_web_pt1:", df_web_pt1.duplicated().sum())
print("Duplicados en df_web_pt2:", df_web_pt2.duplicated().sum())

Duplicados en df_web_pt1: 2095
Duplicados en df_web_pt2: 8669


In [13]:
# 4.2 Eliminar duplicados
df_web_pt1 = df_web_pt1.drop_duplicates()
df_web_pt2 = df_web_pt2.drop_duplicates()

print("Filas df_web_pt1 tras limpiar:", len(df_web_pt1))
print("Filas df_web_pt2 tras limpiar:", len(df_web_pt2))

Filas df_web_pt1 tras limpiar: 341046
Filas df_web_pt2 tras limpiar: 403595


In [14]:
# 4.3 Unir df_web_pt1 y df_web_pt2
df_web = pd.concat([df_web_pt1, df_web_pt2], ignore_index=True)

print("Filas df_web_pt1:", len(df_web_pt1))
print("Filas df_web_pt2:", len(df_web_pt2))
print("Filas df_web total:", len(df_web))

Filas df_web_pt1: 341046
Filas df_web_pt2: 403595
Filas df_web total: 744641


In [15]:
# 4.4 Revisar nulos en df_web
print("Nulos en df_web:")
print(df_web.isna().sum())

Nulos en df_web:
client_id       0
visitor_id      0
visit_id        0
process_step    0
date_time       0
dtype: int64


In [16]:
# 4.5 Convertir date_time a formato datetime para que Python pueda leer bien las fechas y horas
df_web["date_time"] = pd.to_datetime(df_web["date_time"])

print("Tipo de dato antes: object (string)")
print("Tipo de dato después:", df_web["date_time"].dtype)
print()
print("Ejemplo de fechas:")
print(df_web["date_time"].head())

Tipo de dato antes: object (string)
Tipo de dato después: datetime64[us]

Ejemplo de fechas:
0   2017-04-17 15:27:07
1   2017-04-17 15:26:51
2   2017-04-17 15:19:22
3   2017-04-17 15:19:13
4   2017-04-17 15:18:04
Name: date_time, dtype: datetime64[us]


## 5. Guardar datasets limpios

merge

In [17]:
# Guardar datasets limpios
df_demo.to_csv(DATA_PATH / "df_demo_clean.csv", index=False)
df_experiment.to_csv(DATA_PATH / "df_experiment_clean.csv", index=False)
df_web.to_csv(DATA_PATH / "df_web_clean.csv", index=False)

print("Datasets guardados correctamente!")
print("df_demo_clean:      ", len(df_demo), "filas")
print("df_experiment_clean:", len(df_experiment), "filas")
print("df_web_clean:       ", len(df_web), "filas")

Datasets guardados correctamente!
df_demo_clean:       70591 filas
df_experiment_clean: 50500 filas
df_web_clean:        744641 filas
